# LeetCode #1140: Stone Game II

https://leetcode.com/problems/stone-game-ii/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n! )$ | $O(n)$ |
| **Optimal: DP with Suffix Sums ★** | $O(n^2)$ | $O(n^2)$ |

---

## Understanding the Methods

### Brute Force
Simulate every possible sequence of moves recursively without memoization. The number of game states grows factorially, making this infeasible beyond tiny inputs.

### Optimal: DP with Suffix Sums ★
Precompute suffix sums so the total from any position to the end is $O(1)$. Then `dp[i][m]` stores the maximum stones the current player can collect starting at pile `i` with the current multiplier `m`. If a player takes $X$ piles ($1 \leq X \leq 2m$), the opponent plays from `i+X` with `max(m, X)`. The result is the suffix sum minus what the opponent takes (minimax recast as max-of-self).

**Constraints:**
* $1 \leq \text{piles.length} \leq 100$
* $1 \leq \text{piles}[i] \leq 10^4$

## Solutions

### C#

In [ ]:
public class Solution {
    public int StoneGameII(int[] piles) {
        int n = piles.Length;
        // Precompute suffix sums for O(1) total-from-index queries
        int[] suffix = new int[n + 1];
        for (int i = n - 1; i >= 0; i--) suffix[i] = suffix[i + 1] + piles[i];
        int[,] dp = new int[n, n + 1];
        for (int i = 0; i < n; i++)
            for (int m = 1; m <= n; m++) {
                // If we can take all remaining piles, do so
                if (i + 2 * m >= n) { dp[i, m] = suffix[i]; continue; }
                // Try taking x piles; opponent then plays optimally from i+x
                for (int x = 1; x <= 2 * m; x++)
                    dp[i, m] = Math.Max(dp[i, m],
                        suffix[i] - dp[i + x, Math.Max(m, x)]);
            }
        return dp[0, 1];
    }
}

### Python

In [ ]:
class Solution:
    def stone_game_ii(self, piles: list[int]) -> int:
        n = len(piles)
        # Precompute suffix sums for O(1) total-from-index queries
        suffix = [0] * (n + 1)
        for i in range(n - 1, -1, -1):
            suffix[i] = suffix[i + 1] + piles[i]
        dp = [[0] * (n + 1) for _ in range(n)]
        for i in range(n - 1, -1, -1):
            for m in range(1, n + 1):
                # If we can take all remaining piles, do so
                if i + 2 * m >= n:
                    dp[i][m] = suffix[i]
                    continue
                # Try taking x piles; opponent then plays optimally from i+x
                for x in range(1, 2 * m + 1):
                    dp[i][m] = max(dp[i][m], suffix[i] - dp[i + x][max(m, x)])
        return dp[0][1]

### Go

In [ ]:
func stoneGameII(piles []int) int {
    n := len(piles)
    // Precompute suffix sums for O(1) total-from-index queries
    suffix := make([]int, n+1)
    for i := n - 1; i >= 0; i-- { suffix[i] = suffix[i+1] + piles[i] }
    dp := make([][]int, n)
    for i := range dp { dp[i] = make([]int, n+1) }
    for i := n - 1; i >= 0; i-- {
        for m := 1; m <= n; m++ {
            // If we can take all remaining piles, do so
            if i+2*m >= n { dp[i][m] = suffix[i]; continue }
            // Try taking x piles; opponent then plays optimally from i+x
            for x := 1; x <= 2*m; x++ {
                newM := m; if x > m { newM = x }
                candidate := suffix[i] - dp[i+x][newM]
                if candidate > dp[i][m] { dp[i][m] = candidate }
            }
        }
    }
    return dp[0][1]
}

### Rust

In [ ]:
impl Solution {
    pub fn stone_game_ii(piles: Vec<i32>) -> i32 {
        let n = piles.len();
        // Precompute suffix sums for O(1) total-from-index queries
        let mut suffix = vec![0i32; n + 1];
        for i in (0..n).rev() { suffix[i] = suffix[i+1] + piles[i]; }
        let mut dp = vec![vec![0i32; n + 1]; n];
        for i in (0..n).rev() {
            for m in 1..=n {
                // If we can take all remaining piles, do so
                if i + 2 * m >= n { dp[i][m] = suffix[i]; continue; }
                // Try taking x piles; opponent then plays optimally from i+x
                for x in 1..=(2*m) {
                    let new_m = m.max(x);
                    dp[i][m] = dp[i][m].max(suffix[i] - dp[i+x][new_m]);
                }
            }
        }
        dp[0][1]
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `piles = [2, 7, 9, 4, 4]`
Alex starts at index 0 with $m=1$. She can take 1 or 2 piles. Taking 2 (sum $= 9$) gives Lee $m = 2$, and the suffix-sum DP shows Alex ends with $10$ total. The answer is $10$.

### 2. Slightly Complex
**Input:** `piles = [1, 2, 3, 4, 5, 100]`
The dominant pile ($100$) is at the end. Alex must grow $m$ fast enough to reach it before Lee can. The DP identifies the optimal sequence that lets Alex claim the bulk of value.

### 3. Edge Case: Time Factor
**Input:** `piles = [1] * 100`
All piles equal; the DP fills all $n \times n = 10{,}000$ cells. Each cell tries up to $2m$ options, giving the theoretical maximum of $\approx n^3 / 3$ operations — the worst case for computation.

### 4. Edge Case: Space Factor
**Input:** `piles = [10000]`
Single pile; Alex takes it immediately with $m=1$, $2m \geq n$. The DP table has one meaningful cell. Answer is $10{,}000$ — the full pile, with no contention.

### 5. Almost-Impossible but Plausible
**Input:** `piles = [100, 1, 1, ..., 1, 100]` (100 elements)
Valuable piles at both ends force the DP to weigh early greed against growing $m$ to access the far-end pile. The suffix-sum trick makes this tractable even though the naive game tree is exponential.